In [49]:
import os
import numpy as np
from tqdm import tqdm
from collections import Counter, defaultdict
from makam.pitch_extractor import MakamPitchCurve
from raga.pitch_extractor import PitchExtractor
from raga.parser import SwarlipiExtractor

In [50]:
MAKAM_DIR = "datasets/makams"
RAGA_DIR = "datasets/ragas"
TONIC_FREQ = 261.63  # C4

# Sliding N gram
N_GRAM = 10  # N for N-gram analysis
OVERLAP = 9 # N_GRAM-1 is for maximum overlap!

In [51]:
def compute_intervals(pitches):
    """Compute sequential semitone intervals from a pitch sequence."""
    return [pitches[i+1] - pitches[i] for i in range(len(pitches) - 1)]


def extract_makam_intervals(makam_path, tonic_freq=TONIC_FREQ):
    mc = MakamPitchCurve(makam_path, tonic_freq=tonic_freq)
    times, freqs = mc.get_pitch_curve()
    # Convert to semitone intervals from tonic
    rel_pitches = np.array(mc.pitches) - mc.tonic_pitch
    intervals = compute_intervals(rel_pitches)
    return intervals, mc.get_metadata()


def extract_raga_intervals(raga_path, tonic_freq=TONIC_FREQ):
    extractor = PitchExtractor(tonic_freq=tonic_freq)
    curve = extractor.extract_pitch_curve(raga_path)
    # Convert to semitone intervals from tonic (Sa)
    freqs = curve.frequencies
    # Only use non-rests
    mask = freqs > 0
    rel_pitches = 12 * np.log2(freqs[mask] / tonic_freq)
    intervals = compute_intervals(rel_pitches)
    return intervals, curve

def ngram_counter(intervals, n, overlap=OVERLAP):
    # Only count n-grams where not all intervals are zero
    step = max(1, n - overlap)
    return Counter(
        tuple(intervals[i:i+n])
        for i in range(0, len(intervals)-n+1, step)
        if not all(x == 0 for x in intervals[i:i+n])
    )

def cosine_similarity(counter1, counter2):
    all_keys = set(counter1.keys()) | set(counter2.keys())
    v1 = np.array(
        [counter1.get(k, 0) for k in all_keys], dtype=float
    )
    v2 = np.array(
        [counter2.get(k, 0) for k in all_keys], dtype=float
    )
    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    return dot / norm if norm > 0 else 0.0

In [52]:
# Extract 3-grams (Here each gram is notes so 3 gram uses 3 notes)

print(f"Extracting all Raag Bhairav XML files from {RAGA_DIR}")
extractor = SwarlipiExtractor(RAGA_DIR)
bhairav_files = []
for xml_file in extractor.xml_files:
    root = extractor._parse_xml(xml_file)
    if root is not None and extractor._get_raag_name(root).strip().lower() == "bhairav":
        bhairav_files.append(xml_file)
print(f"Found {len(bhairav_files)} Bhairav compositions.")

# --- Aggregate all Bhairav intervals ---
all_bhairav_intervals = []
for xml_file in bhairav_files:
    try:
        intervals, _ = extract_raga_intervals(str(xml_file), tonic_freq=TONIC_FREQ)
        all_bhairav_intervals.extend(intervals)
    except Exception as e:
        print(f"Error processing {xml_file}: {e}")
bhairav_ngrams = ngram_counter(all_bhairav_intervals, N_GRAM)
print(f"Bhairav: {len(all_bhairav_intervals)} intervals, {len(bhairav_ngrams)} unique {N_GRAM}-grams")


Extracting all Raag Bhairav XML files from datasets/ragas
Found 42 Bhairav compositions.
Bhairav: 510297 intervals, 1542 unique 10-grams


In [53]:
import os

makam_results = []

xml_files = [f for f in os.listdir(MAKAM_DIR) if f.endswith(".xml")]

for fname in tqdm(xml_files, desc="Processing Makam files"):
    makam_path = os.path.join(MAKAM_DIR, fname)
    try:
        makam_intervals, meta = extract_makam_intervals(makam_path)
        makam_ngrams = ngram_counter(makam_intervals, N_GRAM)
        sim = cosine_similarity(bhairav_ngrams, makam_ngrams)

        makam_results.append({
            "file": fname,
            "makam": meta.get("makam", ""),
            "usul": meta.get("usul", ""),
            "sim": sim,
            "ngrams": makam_ngrams,
            "intervals": makam_intervals
        })

    except Exception:
        pass

In [54]:
makam_results.sort(key=lambda x: x['sim'], reverse=True)
print("\nMost similar makam to Bhairav:")
for r in makam_results[:3]:
    print(f"  {r['file']} | Makam: {r['makam']} | Usul: {r['usul']} | Similarity: {r['sim']:.4f}")


Most similar makam to Bhairav:
  muhayyerkurdi--sarki--musemmen--ayrilik_var--selahattin_icli.xml | Makam: Muhayyerkürdî Şarkı | Usul: Müsemmen | Similarity: 0.0041
  muhayyerkurdi--sarki--nimsofyan--gonlumu_gonlune--omer_sami_gupgup.xml | Makam: Muhayyerkürdî Şarkı | Usul: Nîmsofyan | Similarity: 0.0030
  suzidil--taksim--serbest---tanburi_cemil_bey.xml | Makam: Sûzidil Taksim | Usul: [Serbest] | Similarity: 0.0027


In [55]:
all_makam_ngrams = Counter()
for r in makam_results:
    all_makam_ngrams.update(r['ngrams'])
print(f"\nMost common {N_GRAM}-grams across all makams:")
for ngram, count in all_makam_ngrams.most_common(10):
    print(f"  {ngram}: {count}x")


Most common 10-grams across all makams:
  (-0.9056603800000005, 0.9056603800000005, -0.9056603800000005, 0.9056603800000005, 2.0377358399999963, 2.0377358500000042, -2.0377358500000042, -2.0377358399999963, -0.9056603800000005, -2.0377358500000042): 35x
  (0.9056603800000005, -0.9056603800000005, -2.0377358500000042, -1.1320754699999895, 1.1320754699999895, 2.0377358500000042, 0.9056603800000005, -0.9056603800000005, -2.0377358500000042, -1.1320754699999895): 30x
  (-1.1320754700000037, -0.9056603800000005, 0.9056603800000005, -0.9056603800000005, 0.9056603800000005, 1.1320754700000037, 2.03773584999999, -2.03773584999999, -1.1320754700000037, 0.0): 30x
  (-2.0377358399999963, -2.0377358500000042, -0.9056603800000005, -2.03773584999999, -2.0377358500000042, 2.0377358500000042, 2.03773584999999, 0.9056603800000005, -0.9056603800000005, -2.03773584999999): 29x
  (-1.584905660000004, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0): 28x
  (0.0, -2.0377358500000042, 0.0, -2.0377358399999963, 

In [56]:
shared_ngrams = set(bhairav_ngrams) & set(all_makam_ngrams)
shared_counts = [(ng, bhairav_ngrams[ng], all_makam_ngrams[ng], bhairav_ngrams[ng] + all_makam_ngrams[ng]) for ng in shared_ngrams]
shared_counts.sort(key=lambda x: x[3], reverse=True)
print(f"\nMost common shared {N_GRAM}-grams (Bhairav + any Makam):")
print(f"{'n-gram':>20} | {'Bhairav':>8} | {'Makams':>8} | {'Total':>8}")
print('-'*55)
for ng, bcount, mcount, total in shared_counts[:10]:
    print(f"{ng!s:>20} | {bcount:8d} | {mcount:8d} | {total:8d}")


Most common shared 10-grams (Bhairav + any Makam):
              n-gram |  Bhairav |   Makams |    Total
-------------------------------------------------------
(0.0, 0.0, 12.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0) |      255 |        3 |      258
(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 12.0) |      255 |        3 |      258
(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 12.0, 0.0) |      255 |        2 |      257
(12.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0) |      255 |        2 |      257
(0.0, 0.0, 0.0, 0.0, 12.0, 0.0, 0.0, 0.0, 0.0, 0.0) |      255 |        1 |      256
(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 12.0, 0.0, 0.0) |      255 |        1 |      256
(0.0, 0.0, 0.0, 0.0, 0.0, 12.0, 0.0, 0.0, 0.0, 0.0) |      255 |        1 |      256
(0.0, 12.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0) |      255 |        1 |      256
